# Derivation of Closest-to-Beamline $z$ and Its Uncertainty

A track at state point $(x_0, y_0, z_0)$ has slopes $(t_x, t_y)$.
We want to find: at the position closest to the beamline $(x,y)=(0,0)$, what is $z_{\text{bl}}$? What is its uncertainty $\sigma_{z_{\text{bl}}}$?

Two approaches:
1. **Direct method**: derive the analytic expression for $z_{\text{bl}}$ and propagate uncertainties
2. **Two independent measurements**: the $xz$ and $yz$ projections each give a $z$ estimate, then combine via weighted average

In [ ]:
from sympy import *
init_printing()

In [ ]:
x, y, z0 = symbols('x y z_0')
tx, ty = symbols('t_x t_y')
dz = symbols('Delta_z')

## Part 1: Finding $z_{\text{bl}}$

The track coordinates at any $z = z_0 + \Delta z$ are:
$$x(\Delta z) = x + t_x \Delta z, \quad y(\Delta z) = y + t_y \Delta z$$

Squared distance to the beamline:
$$r^2(\Delta z) = x(\Delta z)^2 + y(\Delta z)^2$$

Minimize $r^2$: set $\frac{dr^2}{d(\Delta z)} = 0$.

In [ ]:
r2 = (x + tx * dz)**2 + (y + ty * dz)**2

dr2 = diff(r2, dz)
display(Eq(symbols('dr^{2}/d(Deltaz)'), dr2))

dz_bl = solve(dr2, dz)[0]
display(Eq(symbols('Deltaz_{bl}'), dz_bl))

Result:
$$\boxed{z_{\text{bl}} = z_0 - \frac{t_x x + t_y y}{t_x^2 + t_y^2}}$$

---
## Part 2: Uncertainty $\sigma_{z_{\text{bl}}}$ — Method 1: Direct Error Propagation

$z_{\text{bl}}$ is a function of $(x, y, t_x, t_y)$. Define $n = t_x^2 + t_y^2$.

$$\sigma^2_{z_{\text{bl}}} = \sum_{i,j} \frac{\partial (\Delta z)}{\partial p_i} \frac{\partial (\Delta z)}{\partial p_j} \, \text{cov}(p_i, p_j)$$

where $p_i, p_j$ are the track state parameters, $\vec{p} = (x, \, y, \, t_x, \, t_y)$:
- $x, y$: transverse coordinates at the reference point $z_0$
- $t_x = dx/dz, \, t_y = dy/dz$: track slopes in the $xz$ and $yz$ planes

The double sum runs over all $4 \times 4$ parameter pairs, with $\text{cov}(p_i, p_j)$ from the track fit covariance matrix.

In [ ]:
n = tx**2 + ty**2
params = [x, y, tx, ty]
labels = ['x', 'y', 't_x', 't_y']

J = [simplify(diff(dz_bl, p)) for p in params]

for label, ji in zip(labels, J):
    display(Eq(Symbol(f'partial(Deltaz)/partial({label})'), ji))

# At the closest-to-beam point, the state is evaluated there; perpendicularity condition: tx*x + ty*y = 0
print('\nAt the closest-to-beam point (t_x·x + t_y·y = 0):')
J_at_bl = [simplify(ji.subs(tx*x + ty*y, 0)) for ji in J]
for label, ji in zip(labels, J_at_bl):
    display(Eq(Symbol(f'partial(Deltaz)/partial({label})'), ji))

That is:
$$\frac{\partial(\Delta z)}{\partial x} = -\frac{t_x}{n}, \quad \frac{\partial(\Delta z)}{\partial y} = -\frac{t_y}{n}$$

$$\frac{\partial(\Delta z)}{\partial t_x} = \frac{2t_x(t_x x + t_y y) - x \, n}{n^2}, \quad \frac{\partial(\Delta z)}{\partial t_y} = \frac{2t_y(t_x x + t_y y) - y \, n}{n^2}$$

**Perpendicularity condition**: at the closest-to-beam point, the distance $r$ from the track to the beamline is minimized, i.e. $dr^2/d(\Delta z) = 0$. This is equivalent to saying that the vector from the beamline to the track is perpendicular to the track direction $(t_x, t_y)$:
$$t_x \, x_{\text{bl}} + t_y \, y_{\text{bl}} = 0$$
If we choose the track state reference point to be the closest-to-beam point itself (i.e. $(x, y)$ are the coordinates at that point), then $x = x_{\text{bl}}, \, y = y_{\text{bl}}$, and the above becomes:
$$t_x \, x + t_y \, y = 0$$
Substituting into the slope partial derivatives simplifies them to:

$$\frac{\partial(\Delta z)}{\partial t_x} = -\frac{x}{n}, \quad \frac{\partial(\Delta z)}{\partial t_y} = -\frac{y}{n}$$

The four partial derivatives have a nice symmetry: $x \leftrightarrow t_x$, $y \leftrightarrow t_y$.

### Simplification: assuming $xz$ and $yz$ planes are independent

The track parameters $(x, t_x)$ in the $xz$ plane come from the same fit, so they have a non-zero covariance $\text{cov}(x, t_x) \neq 0$; likewise for $(y, t_y)$ in the $yz$ plane. However, the two planes are fitted independently, so:

$$\text{cov}(x, y) = \text{cov}(x, t_y) = \text{cov}(y, t_x) = \text{cov}(t_x, t_y) = 0$$

The covariance matrix then has a block-diagonal form:

$$C = \begin{pmatrix} V_{xx} & 0 & V_{x t_x} & 0 \\ 0 & V_{yy} & 0 & V_{y t_y} \\ V_{x t_x} & 0 & V_{t_x t_x} & 0 \\ 0 & V_{y t_y} & 0 & V_{t_y t_y} \end{pmatrix}$$

Error propagation: $\sigma^2_{\Delta z} = J^T C \, J$, where $J$ is the Jacobian vector.

In [ ]:
sigma_x, sigma_y, sigma_tx, sigma_ty = symbols(
    'sigma_x sigma_y sigma_{t_x} sigma_{t_y}', positive=True
)
cov_xtx = Symbol('cov_{x,tx}')
cov_yty = Symbol('cov_{y,ty}')

# Block-diagonal covariance matrix (xz and yz planes independent)
C = Matrix([
    [sigma_x**2, 0,          cov_xtx,     0          ],
    [0,          sigma_y**2, 0,           cov_yty    ],
    [cov_xtx,    0,          sigma_tx**2, 0          ],
    [0,          cov_yty,    0,           sigma_ty**2],
])
print('C =')
display(C)

# Jacobian simplified using perpendicularity condition
J_bl = [simplify(ji.subs(tx*x + ty*y, 0)) for ji in J]
Jvec = Matrix(J_bl)
print('\nJ (at closest-to-beam point) =')
display(Jvec.T)

# Error propagation: sigma^2 = J^T C J, term by term
print(f'\nsigma^2_{{z_bl}} = J^T C J, expanded term by term:\n')
sigma2_terms = []
for i in range(4):
    for j in range(4):
        if C[i, j] != 0:
            coeff = simplify(Jvec[i] * Jvec[j])
            term = coeff * C[i, j]
            sigma2_terms.append(term)
            print(f'({labels[i]}, {labels[j]}):')
            display(term)
            print()

sigma2_full = simplify(Add(*sigma2_terms))
print('Sum:')
display(sigma2_full)

# Further simplification: neglect slope uncertainties
# cov(x, tx) = 0, sigma_tx^2 = 0, likewise for y
print('\nFurther assuming cov(x,tx)=0, sigma^2_tx=0, cov(y,ty)=0, sigma^2_ty=0:')
sigma2_simple = sigma2_full.subs([
    (cov_xtx, 0), (cov_yty, 0),
    (sigma_tx, 0), (sigma_ty, 0),
])
sigma2_simple = simplify(sigma2_simple)
display(sigma2_simple)

---
## Part 3: Uncertainty $\sigma_{z_{\text{bl}}}$ — Method 2: Two Independent Measurements

Alternative approach: the track projections onto the $xz$ and $yz$ planes each independently provide a $z$ estimate.

### Measurement from the $xz$ plane

In the $xz$ plane, the track is $x(z) = x_0 + t_x(z - z_0)$. Setting $x_0 = 0$ gives an estimate of $z$:

$$z = z_0 - \frac{x_0}{t_x}$$

Uncertainty: $\frac{\partial z}{\partial x} = -1/t_x$; 

$$\sigma^2_{z_x} \approx \frac{\sigma^2_x}{t_x^2}$$

### Measurement from the $yz$ plane

Likewise: $z_y = z_0 - y/t_y$,

$$\sigma^2_{z_y} \approx \frac{\sigma^2_y}{t_y^2}$$

### Optimal weighted combination

For two independent measurements $z_x, z_y$, the optimal weights (minimizing combined variance) are $w_i = 1/\sigma^2_{z_i}$:

$$w_x = \frac{t_x^2}{\sigma_x^2}, \quad w_y = \frac{t_y^2}{\sigma_y^2}$$

Combined variance:

$$\frac{1}{\sigma^2_{z_{\text{bl}}}} = w_x + w_y = \frac{t_x^2}{\sigma_x^2} + \frac{t_y^2}{\sigma_y^2}$$

$$\boxed{\sigma^2_{z_{\text{bl}}} = \frac{\sigma_x^2 \sigma_y^2}{t_x^2 \sigma_y^2 + t_y^2 \sigma_x^2}}$$

When $\sigma_x = \sigma_y = \sigma_0$:

$$\boxed{\sigma^2_{z_{\text{bl}}} = \frac{\sigma_0^2}{t_x^2 + t_y^2}}$$

---
## Equivalence of the Two Methods

Method 1 (direct error propagation, neglecting slope uncertainties) gives:

$$\sigma^2_{\text{Method 1}} = \frac{t_x^2 \sigma_x^2 + t_y^2 \sigma_y^2}{(t_x^2 + t_y^2)^2}$$

Method 2 (optimal weighted combination) gives:

$$\sigma^2_{\text{Method 2}} = \frac{\sigma_x^2 \sigma_y^2}{t_x^2 \sigma_y^2 + t_y^2 \sigma_x^2}$$

> **When $\sigma_x = \sigma_y$, the two are strictly equivalent, both yielding $\sigma_0^2 / (t_x^2 + t_y^2)$.**
>
> **When $\sigma_x \neq \sigma_y$, Method 2 (optimal weighting) gives a smaller variance and is the better estimator.**